# QArray manipulation utilities in a batched Dynamiqs workflow

This notebook demonstrates how the QArray manipulation utilities (`swapaxes`, `moveaxis`, `expand_dims`, `where`, and `concatenate`) fit into a realistic Dynamiqs workflow.

The example is a small batched Rabi-oscillation sweep for a qubit. We solve many combinations of detuning, drive strength, and initial state, then reorganize and post-process the resulting `QArray` directly—without converting to raw `jax.Array` objects.

## Research notes from the Dynamiqs API

Dynamiqs represents quantum objects as `QArray` instances:

- the last two axes are the quantum object axes, such as `(n, 1)` for kets and `(n, n)` for operators or density matrices,
- all axes before the last two are batch axes,
- batched solvers return `QArray` results, for example `sesolve(...).states` has shape `(...H, ...psi0, ntsave, n, 1)` with Cartesian batching enabled,
- preserving the last two quantum axes means array manipulations can usually preserve the `QArray` abstraction.

That makes the new utilities especially useful after simulations: batch axes can be reordered, extra scenario axes can be inserted, filtered state banks can be built, and compatible collections can be concatenated while keeping `QArray` semantics.

In [ ]:
mport jax.numpy as jnp
import matplotlib.pyplot as plt

import dynamiqs as dq

# Keep notebook output compact.
dq.set_progress_meter(False)

## 1. Build a batched qubit problem

We create a simple ZX Hamiltonian sweep:

$$
H(\Delta, \Omega) = \Delta\sigma_z + \Omega\sigma_x.
$$

The Hamiltonian batch axes are `(detuning, drive)`. We also prepare four initial states, giving an additional initial-state batch axis.

In [ ]:
detunings = jnp.linspace(-1.0, 1.0, 3)  # detuning batch axis
drives = jnp.linspace(0.5, 1.5, 4)  # drive-strength batch axis

H = (
    detunings[:, None, None, None] * dq.sigmaz()
    + drives[None, :, None, None] * dq.sigmax()
)

g = dq.ground()
e = dq.excited()
plus = (g + e).unit()
minus = (g - e).unit()
psi0 = dq.stack([g, e, plus, minus])

print(f'H shape:    {H.shape}    # (detuning, drive, n, n)')
print(f'psi0 shape: {psi0.shape}       # (initial_state, n, 1)')
print(f'H dims: {H.dims}, psi0 dims: {psi0.dims}')

## 2. Solve all combinations in one batched call

With the default Cartesian batching, the result combines all Hamiltonian batches with the initial-state batch:

`(detuning, drive, initial_state, time, n, 1)`.

In [ ]:
tsave = jnp.linspace(0.0, 2.0, 21)
result = dq.sesolve(H, psi0, tsave, exp_ops=[dq.sigmaz()], progress_meter=False)
states = result.states

print(f'states type:  {type(states).__name__}')
print(f'states shape: {states.shape}')
print('axis meaning: (detuning, drive, initial_state, time, n, 1)')

## 3. Reorder batch axes with `moveaxis` and `swapaxes`

Here we put `initial_state` first, then produce a drive-first view.

Because the quantum axes remain the last two dimensions, both manipulations return `QArray` objects.

In [ ]:
# Move initial_state from axis 2 to axis 0:
# (detuning, drive, initial_state, time, n, 1)
# -> (initial_state, detuning, drive, time, n, 1)
states_by_initial = dq.moveaxis(states, 2, 0)

# Swap detuning and drive axes:
# -> (initial_state, drive, detuning, time, n, 1)
states_drive_first = dq.swapaxes(states_by_initial, 1, 2)

print(
    f'states_by_initial: type={type(states_by_initial).__name__}, '
    f'shape={states_by_initial.shape}'
)
print(
    f'states_drive_first: type={type(states_drive_first).__name__}, '
    f'shape={states_drive_first.shape}'
)

## 4. Add scenario axes with `expand_dims`

`expand_dims` is useful when building higher-level result banks. Here we add a leading `scenario` axis while preserving the qarray quantum axes.

In [ ]:
scenario_states = dq.expand_dims(states_by_initial, 0)

print(f'scenario_states type:  {type(scenario_states).__name__}')
print(f'scenario_states shape: {scenario_states.shape}')
print('axis meaning: (scenario, initial_state, detuning, drive, time, n, 1)')

## 5. Filter states with `where`

At the final time, let us compute the excited-state population. We then keep the final state only when the excited population is above a threshold. Otherwise we replace it with the ground state.

The condition is a regular JAX boolean array, while `x` and `y` are `QArray` objects. The result remains a `QArray` because the selected values have compatible Hilbert-space dimensions.

In [ ]:
final_states = states_by_initial[..., -1, :, :]
excited_projector = dq.excited().proj()
final_excited_pop = dq.expect(excited_projector, final_states).real

# Broadcast |g> to the same batch shape as final_states.
ground_like = dq.expand_dims(dq.ground(), (0, 1, 2)).broadcast_to(*final_states.shape)

selected_final_states = dq.where(
    final_excited_pop[..., None, None] > 0.5, final_states, ground_like
)

print(f'final_states shape:          {final_states.shape}')
print(f'final_excited_pop shape:     {final_excited_pop.shape}')
print(f'selected_final_states type:  {type(selected_final_states).__name__}')
print(f'selected_final_states shape: {selected_final_states.shape}')

## 6. Combine compatible QArray collections with `concatenate`

Finally, build a two-scenario bank: the raw final states and the threshold-filtered final states. Because we concatenate along a batch axis and keep the last two quantum axes intact, the output is still a `QArray`.

In [ ]:
final_state_bank = dq.concatenate(
    [dq.expand_dims(final_states, 0), dq.expand_dims(selected_final_states, 0)], axis=0
)

print(f'final_state_bank type:  {type(final_state_bank).__name__}')
print(f'final_state_bank shape: {final_state_bank.shape}')
print('axis meaning: (scenario, initial_state, detuning, drive, n, 1)')

## 7. A quick visualization

The manipulation utilities let us organize data before plotting. Below we plot the final excited-state population for the `|g>` initial state, with detuning on rows and drive strength on columns.

In [ ]:
ground_initial_index = 0
heatmap = final_excited_pop[ground_initial_index]

fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(heatmap, origin='lower', aspect='auto')
ax.set_xticks(range(len(drives)), [f'{x:.2f}' for x in drives])
ax.set_yticks(range(len(detunings)), [f'{x:.2f}' for x in detunings])
ax.set_xlabel('Drive strength Ω')
ax.set_ylabel('Detuning Δ')
ax.set_title('Final excited-state population from |g>')
fig.colorbar(im, ax=ax, label='Population')
plt.show()

## Takeaway

For normal Dynamiqs users' workflows, these utilities are most useful on **batch axes**:

- `moveaxis` and `swapaxes` reorganize simulation outputs for analysis;
- `expand_dims` adds scenario or ensemble axes;
- `where` builds filtered or conditionally selected state collections;
- `concatenate` combines compatible batches.

When the last two quantum axes remain valid, outputs preserve the `QArray` abstraction and can be passed directly into other Dynamiqs utilities such as `dq.expect`, `dq.to_qutip`, or other additional simulations.